# 01 — Veri toplama ve kalite denetimi

Bu defter gerçek EVDS makro verisini ve yfinance günlük endeks kapanışlarından hazırlanmış yerel girdiyi denetler. Ham dosyalar `data/private/` altında kalır; depoda yalnızca türetilmiş araştırma sonuçları bulunur.

## Araştırma kapsamı

- Ocak 2019–Aralık 2024
- Gecikmeli hesaplamalar için Aralık 2017’den başlayan günlük kapanışlar
- XBANK bankacılık, XUSIN sanayi temsilcisidir
- Aylık getiri ay sonu kapanışlarından; oynaklık günlük log getirilerden hesaplanır.

In [1]:
from pathlib import Path
import json, tomllib
import pandas as pd
import plotly.express as px
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "config.toml").exists() else Path.cwd().parent
RAW, OUT = ROOT / "data/private/study-yahoo-real", ROOT / "results/research"
assert RAW.exists(), "Gerçek ham girdi data/private altında hazırlanmalıdır."
config = tomllib.loads((ROOT / "config.toml").read_text(encoding="utf-8"))
START, END, SECTORS = pd.Timestamp(config["study"]["start"]), pd.Timestamp(config["study"]["end"]), config["study"]["sectors"]
print(f"Çalışma dönemi: {START.date()} — {END.date()}")
print("Temsilciler:", ", ".join(SECTORS))

Çalışma dönemi: 2019-01-01 — 2024-12-31
Temsilciler: XBANK, XUSIN


In [2]:
from bist_risk.data import read_inputs, monthly_prices, align_macro, shock_scores
prices, macro, calendar, provenance = read_inputs(RAW, "research")
monthly = monthly_prices(prices, calendar, config["study"]["min_days"])
aligned = align_macro(macro, pd.date_range(monthly.date.min(), END, freq="ME"))
study_dates = pd.date_range(START, END, freq="ME")
quality_rows, eligible = [], []
for sector in SECTORS:
    block = monthly[(monthly.sector == sector) & monthly.date.between(START, END)].set_index("date").reindex(study_dates)
    complete = len(block) == len(study_dates) and block[["return_value", "volatility"]].notna().all().all()
    quality_rows.append({"sector": sector, "eligible": bool(complete), "months": int(block.return_value.count()), "reason": "complete" if complete else "missing daily sessions, warm-up or monthly data"})
    if complete: eligible.append(sector)
quality = pd.DataFrame(quality_rows)
assert eligible == SECTORS, quality

In [3]:
display(quality)
coverage = prices.groupby("sector").agg(first_date=("date","min"), last_date=("date","max"), daily_sessions=("date","count"))
display(coverage)
print("Makro seri sayısı:", macro.series.nunique(), "| İşlem seansı:", calendar.date.nunique())

,sector,eligible,months,reason
0,XBANK,True,72,complete
1,XUSIN,True,72,complete


,first_date,last_date,daily_sessions
sector,,,
XBANK,2017-12-01,2024-12-31,1758
XUSIN,2017-12-01,2024-12-31,1758


Makro seri sayısı: 6 | İşlem seansı: 1758


In [4]:
sources = pd.DataFrame(provenance["series"]).T[["source_url", "frequency", "unit", "transformation", "vintage_policy"]]
display(sources)

,source_url,frequency,unit,transformation,vintage_policy
XBANK,https://finance.yahoo.com/quote/XBANK.IS/history/,daily,BIST price index points,Yahoo Finance unadjusted Close; no imputation,latest-available download; not an official BIS...
XUSIN,https://finance.yahoo.com/quote/XUSIN.IS/history/,daily,BIST price index points,Yahoo Finance unadjusted Close; no imputation,latest-available download; not an official BIS...
usdtry,https://evds3.tcmb.gov.tr/igmevdsms-dis/,daily,TRY per USD,EVDS TP.DK.USD.A; month-end; availability guar...,latest revised download with conservative cale...
brent,https://evds3.tcmb.gov.tr/igmevdsms-dis/,daily,USD per barrel,EVDS TP.BRENTPETROL.EUBP; month-end; availabil...,latest revised download with conservative cale...
cpi,https://evds3.tcmb.gov.tr/igmevdsms-dis/,monthly,index (2003=100),EVDS TP.GENENDEKS.T1; last; availability guard...,latest revised download with conservative cale...
industrial_production,https://evds3.tcmb.gov.tr/igmevdsms-dis/,monthly,index (2021=100),EVDS TP.TSANAYMT2021.BCD; last; availability g...,latest revised download with conservative cale...
policy_rate,https://evds3.tcmb.gov.tr/igmevdsms-dis/,monthly,percent,EVDS TP.BISPOLFAIZ.TUR; last; availability gua...,latest revised download with conservative cale...
gdp_growth,https://evds3.tcmb.gov.tr/igmevdsms-dis/,quarterly,percent quarter-on-quarter,EVDS TP.GSYIH60.HY.B1GQ; qoq change from chain...,latest revised download with conservative cale...


In [5]:
study_monthly = monthly[monthly.date.between(START, END)]
px.line(study_monthly, x="date", y="close", color="sector", title="Gerçek veri — aylık endeks kapanışları", labels={"close":"Endeks puanı", "date":""}).show()
px.line(study_monthly, x="date", y="volatility", color="sector", title="Gerçek veri — yıllıklaştırılmış gerçekleşen oynaklık", labels={"volatility":"Oynaklık", "date":""}).show()

In [6]:
availability = macro.assign(delay_days=(macro.available_at-macro.period).dt.days).groupby("series").agg(observations=("value","count"), first_period=("period","min"), last_period=("period","max"), availability_guard_days=("delay_days","max"))
display(availability)
assert (macro.available_at >= macro.period).all()
print("Yayın tarihi koruma denetimi geçti.")

,observations,first_period,last_period,availability_guard_days
series,,,,
brent,85,2017-12-31,2024-12-31,0
cpi,85,2017-12-31,2024-12-31,35
gdp_growth,28,2018-03-31,2024-12-31,100
industrial_production,85,2017-12-31,2024-12-31,60
policy_rate,85,2017-12-31,2024-12-31,35
usdtry,85,2017-12-31,2024-12-31,0


Yayın tarihi koruma denetimi geçti.


In [7]:
from bist_risk.artifacts import write_tables
write_tables(OUT, {"monthly":study_monthly, "macro":aligned.loc[START:END].reset_index(), "shocks":shock_scores(aligned).loc[START:END].reset_index(), "quality":quality})
print("01 sonuçları results/research altında kaydedildi.")

01 sonuçları results/research altında kaydedildi.
